# Python — Parte 5: Pandas

**Conteúdo:** DataFrame · seleção · filtros · índices · dados faltantes (`NaN`) · leitura e gravação de CSV e Excel

O **pandas** organiza dados em tabelas (linhas × colunas) e é amplamente utilizado na análise de dados em Python.

- **Series** → uma coluna (um vetor com rótulos).
- **DataFrame** → uma tabela inteira (várias Series lado a lado).

> **Arquivos de demonstração:** neste notebook, os exemplos de CSV, TXT e Excel são gerados automaticamente em `outputs/` e reutilizados nas células seguintes. Execute as células em ordem. A pasta `datasets/` é reservada para bases externas que você acrescentar no futuro. Nenhum upload é necessário para os exemplos atuais.


## 1. Criando um DataFrame

Ingredientes: os **valores** (matriz), o **índice** (nome das linhas) e as **colunas** (nome das colunas).

In [1]:
import numpy as np
import pandas as pd

np.random.seed(123)
valores = np.random.randn(5, 4)          # matriz 5x4 de números aleatórios (normal)
df = pd.DataFrame(valores, index=['a', 'b', 'c', 'd', 'e'], columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
a,-1.085631,0.997345,0.282978,-1.506295
b,-0.578600,1.651437,-2.426679,-0.428913
c,1.265936,-0.866740,-0.678886,-0.094709
d,1.491390,-0.638902,-0.443982,-0.434351
e,2.205930,2.186786,1.004054,0.386186


Também dá para criar a partir de um **dicionário** (cada chave vira uma coluna) ou de uma **lista de dicionários** (cada dicionário vira uma linha).

In [2]:
df_dict = pd.DataFrame({'nome': ['Ana', 'Bruno', 'Carla'],
                        'idade': [30, 25, 35],
                        'cidade': ['Campinas', 'Santos', 'Campinas']})
df_dict

,nome,idade,cidade
0,Ana,30,Campinas
1,Bruno,25,Santos
2,Carla,35,Campinas


In [3]:
pessoas = [{'nome': 'Ana', 'idade': 30}, {'nome': 'Bruno', 'idade': 25}]
pd.DataFrame(pessoas)

,nome,idade
0,Ana,30
1,Bruno,25


## 2. Selecionando dados

| Quero... | Use |
|---|---|
| uma coluna | `df['A']` (devolve uma Series) |
| várias colunas | `df[['A', 'C']]` (lista dentro dos colchetes) |
| linha/célula **pelo nome** | `df.loc['b']` · `df.loc['b', 'C']` |
| linha/célula **pela posição** | `df.iloc[1]` · `df.iloc[1, 2]` |

In [4]:
print(df['A'])
print()
print(df[['A', 'C']])

a   -1.085631
b   -0.578600
c    1.265936
d    1.491390
e    2.205930
Name: A, dtype: float64

          A         C
a -1.085631  0.282978
b -0.578600 -2.426679
c  1.265936 -0.678886
d  1.491390 -0.443982
e  2.205930  1.004054


In [5]:
print(df.loc['b'])                 # linha 'b' inteira
print()
print(df.loc['b', 'C'])            # linha 'b', coluna 'C'
print(df.loc['a':'c', ['A', 'B']]) # no loc o fim da fatia É incluído
print()
print(df.iloc[1, 2])               # 2ª linha, 3ª coluna (por posição)
print(df.iloc[0:2, 0:2])           # no iloc o fim NÃO é incluído

A   -0.578600
B    1.651437
C   -2.426679
D   -0.428913
Name: b, dtype: float64

-2.426679243393074
          A         B
a -1.085631  0.997345
b -0.578600  1.651437
c  1.265936 -0.866740

-2.426679243393074
          A         B
a -1.085631  0.997345
b -0.578600  1.651437


## 3. Filtrando com condições

Assim como no NumPy, uma comparação gera uma máscara de `True`/`False`.

In [6]:
df > 0                       # máscara com o mesmo formato da tabela

,A,B,C,D
a,False,True,True,False
b,False,True,False,False
c,True,False,False,False
d,True,False,False,False
e,True,True,True,True


In [7]:
mascara = df > 0
df[mascara]                  # onde é False, aparece NaN

,A,B,C,D
a,NaN,0.997345,0.282978,NaN
b,NaN,1.651437,NaN,NaN
c,1.265936,NaN,NaN,NaN
d,1.491390,NaN,NaN,NaN
e,2.205930,2.186786,1.004054,0.386186


### Filtrando linhas pelo valor de uma coluna
Passando a máscara de **uma coluna**, ficam só as linhas onde ela é `True`.

In [8]:
df[df['A'] > 0]              # só as linhas em que a coluna A é positiva

,A,B,C,D
c,1.265936,-0.866740,-0.678886,-0.094709
d,1.491390,-0.638902,-0.443982,-0.434351
e,2.205930,2.186786,1.004054,0.386186


### Combinando condições
- Use **`&`** no lugar de `and`, **`|`** no lugar de `or` e **`~`** para negar.
- ⚠️ Cada condição precisa estar entre **parênteses**.

In [9]:
print('A > 0  E  C > 1:')
display(df[(df['A'] > 0) & (df['C'] > 1)])

print('A > 0  OU  C > 1:')
display(df[(df['A'] > 0) | (df['C'] > 1)])

print('NÃO (A > 0):')
display(df[~(df['A'] > 0)])

A > 0  E  C > 1:


,A,B,C,D
e,2.20593,2.186786,1.004054,0.386186


A > 0  OU  C > 1:


,A,B,C,D
c,1.265936,-0.866740,-0.678886,-0.094709
d,1.491390,-0.638902,-0.443982,-0.434351
e,2.205930,2.186786,1.004054,0.386186


NÃO (A > 0):


,A,B,C,D
a,-1.085631,0.997345,0.282978,-1.506295
b,-0.578600,1.651437,-2.426679,-0.428913


## 4. Trabalhando com o índice

- `reset_index()` → transforma o índice em uma **coluna** e cria um índice numérico 0, 1, 2...
- `set_index('coluna')` → faz o caminho inverso: uma coluna vira o índice.
- `df['nova'] = ...` → cria (ou substitui) uma coluna.

⚠️ `reset_index()` e `set_index()` **devolvem uma nova tabela** e não alteram a original, a menos que você guarde o resultado (`df = df.set_index(...)`).

In [10]:
df.reset_index()

,index,A,B,C,D
0,a,-1.085631,0.997345,0.282978,-1.506295
1,b,-0.578600,1.651437,-2.426679,-0.428913
2,c,1.265936,-0.866740,-0.678886,-0.094709
3,d,1.491390,-0.638902,-0.443982,-0.434351
4,e,2.205930,2.186786,1.004054,0.386186


In [11]:
col = 'V1 V2 V3 V4 V5'.split()       # .split() separa o texto em uma lista
print(col)

df['val'] = col                      # nova coluna
df

['V1', 'V2', 'V3', 'V4', 'V5']


,A,B,C,D,val
a,-1.085631,0.997345,0.282978,-1.506295,V1
b,-0.578600,1.651437,-2.426679,-0.428913,V2
c,1.265936,-0.866740,-0.678886,-0.094709,V3
d,1.491390,-0.638902,-0.443982,-0.434351,V4
e,2.205930,2.186786,1.004054,0.386186,V5


In [12]:
df.set_index('val')                  # a coluna 'val' vira o índice

,A,B,C,D
val,,,,
V1,-1.085631,0.997345,0.282978,-1.506295
V2,-0.578600,1.651437,-2.426679,-0.428913
V3,1.265936,-0.866740,-0.678886,-0.094709
V4,1.491390,-0.638902,-0.443982,-0.434351
V5,2.205930,2.186786,1.004054,0.386186


In [13]:
df               # a original continua igual (ainda tem a coluna 'val')

,A,B,C,D,val
a,-1.085631,0.997345,0.282978,-1.506295,V1
b,-0.578600,1.651437,-2.426679,-0.428913,V2
c,1.265936,-0.866740,-0.678886,-0.094709,V3
d,1.491390,-0.638902,-0.443982,-0.434351,V4
e,2.205930,2.186786,1.004054,0.386186,V5


## 5. Dados faltantes (`NaN`)

Dados reais quase sempre têm buracos. O pandas representa valor ausente como `NaN`.

| Quero... | Use |
|---|---|
| ver onde faltam dados | `df.isna()` · `df.isna().sum()` |
| remover **linhas** com algum `NaN` | `df.dropna()` |
| remover **colunas** com algum `NaN` | `df.dropna(axis=1)` |
| preencher com um valor | `df.fillna(valor)` |
| preencher com a média da coluna | `df.fillna(df.mean())` |
| repetir o valor anterior / seguinte | `df.ffill()` / `df.bfill()` |

Nenhum deles altera a tabela original (a menos que você guarde o resultado).

In [14]:
import numpy as np
import pandas as pd

d = {'A': [1, 2, np.nan],
     'B': [5, np.nan, np.nan],
     'C': [1, 2, 3]}
df_nan = pd.DataFrame(d)
df_nan

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2
2,NaN,NaN,3


In [15]:
print('Quantos NaN em cada coluna:')
print(df_nan.isna().sum())

Quantos NaN em cada coluna:
A    1
B    2
C    0
dtype: int64


In [16]:
print('dropna() -> remove LINHAS com NaN:')
display(df_nan.dropna())

print('dropna(axis=1) -> remove COLUNAS com NaN:')
display(df_nan.dropna(axis=1))

dropna() -> remove LINHAS com NaN:


,A,B,C
0,1.0,5.0,1


dropna(axis=1) -> remove COLUNAS com NaN:


,C
0,1
1,2
2,3


In [17]:
print('fillna(1):')
display(df_nan.fillna(value=1))

print('fillna(média de cada coluna):')
display(df_nan.fillna(value=df_nan.mean()))

print('ffill() -> repete o valor da linha de cima:')
display(df_nan.ffill())

fillna(1):


,A,B,C
0,1.0,5.0,1
1,2.0,1.0,2
2,1.0,1.0,3


fillna(média de cada coluna):


,A,B,C
0,1.0,5.0,1
1,2.0,5.0,2
2,1.5,5.0,3


ffill() -> repete o valor da linha de cima:


,A,B,C
0,1.0,5.0,1
1,2.0,5.0,2
2,2.0,5.0,3


In [18]:
# Preenchendo com texto: o dado deixa de ser numérico (vira "object")
df_nan.astype(object).fillna('vazio')

,A,B,C
0,1.0,5.0,1
1,2.0,vazio,2
2,vazio,vazio,3


> 💡 Nas versões antigas do pandas usava-se `df.fillna(method='ffill')`. Esse parâmetro foi removido: use `df.ffill()` e `df.bfill()`.

## 6. Lendo e gravando arquivos CSV / TXT

Primeiro, vamos **criar arquivos tabulares de exemplo** na pasta `outputs/`. Embora os dados sejam fictícios, eles são usados como entrada nas demonstrações seguintes. Os nomes indicam o conteúdo de cada arquivo.


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

pasta_outputs = Path('outputs')
pasta_outputs.mkdir(parents=True, exist_ok=True)
np.random.seed(7)

# produtos e estoque: separado por vírgula
pd.DataFrame({'produto': ['caneta', 'caderno', 'lápis', 'borracha', 'régua', 'mochila', 'estojo'],
              'preco': [2.5, 15.9, 1.2, 0.9, 3.5, 89.9, 22.0],
              'estoque': [100, 40, 250, 180, 60, 15, 35]}
             ).to_csv('outputs/produtos_estoque.csv', index=False)

# cidades: separado por ponto e vírgula (comum em arquivos gerados no Brasil)
pd.DataFrame({'cidade': ['Campinas', 'Santos', 'Sorocaba', 'Bauru'],
              'populacao': [1213792, 433656, 723682, 379297],
              'area_km2': [794.4, 280.7, 450.4, 667.7]}
             ).to_csv('outputs/cidades_ponto_virgula.csv', index=False, sep=';')

# matriz numérica: só números
pd.DataFrame(np.random.randint(1, 10, (5, 4)), columns=list('ABCD')
             ).to_csv('outputs/matriz_numerica.csv', index=False)

# vendas: com acentos e salvo em cp1252 (Windows), como muitos arquivos do Excel
n = 40
vendas = pd.DataFrame({
    'Segmento': np.random.choice(['Governo', 'Pequenas empresas', 'Empresas'], n),
    'País': np.random.choice(['Brasil', 'Canadá', 'França', 'Alemanha'], n),
    'Produto': np.random.choice(['Carretera', 'Montana', 'Paseo'], n),
    'Unidades': np.random.randint(100, 3000, n),
    'Preço': np.random.choice([10, 20, 35, 125, 350], n),
    'Mês': np.random.randint(1, 13, n),
})
vendas['Receita'] = vendas['Unidades'] * vendas['Preço']
vendas.to_csv('outputs/vendas_simuladas.csv', index=False, encoding='cp1252')

print('Arquivos gerados:')
for arquivo in sorted(pasta_outputs.iterdir()):
    print('-', arquivo.name)

Arquivos gerados:
- .gitkeep
- cidades_ponto_virgula.csv
- matriz_numerica.csv
- matriz_transformada.csv
- movimentacoes_financeiras.xlsx
- movimentacoes_tratadas.xlsx
- planilha_multiplas_abas.xlsx
- produtos_estoque.csv
- receitas_despesas.xlsx
- resultado.xlsx
- vendas_simuladas.csv


### `read_csv`
- `head(n)` → primeiras linhas · `tail(n)` → últimas · `sample(n)` → linhas aleatórias
- `sep=';'` → quando o separador não é vírgula

In [20]:
df1 = pd.read_csv('outputs/produtos_estoque.csv')
df1.head(2)

,produto,preco,estoque
0,caneta,2.5,100
1,caderno,15.9,40


In [21]:
df1.sample(3)

,produto,preco,estoque
6,estojo,22.0,35
2,lápis,1.2,250
0,caneta,2.5,100


In [22]:
df1.tail()

,produto,preco,estoque
2,lápis,1.2,250
3,borracha,0.9,180
4,régua,3.5,60
5,mochila,89.9,15
6,estojo,22.0,35


Se o separador não for informado, o pandas junta tudo numa coluna só. Por isso usamos `sep=';'`:

In [23]:
df2 = pd.read_csv('outputs/cidades_ponto_virgula.csv', sep=';')
df2.head()

,cidade,populacao,area_km2
0,Campinas,1213792,794.4
1,Santos,433656,280.7
2,Sorocaba,723682,450.4
3,Bauru,379297,667.7


### Conhecendo a tabela: `info()` e `shape`
`shape` devolve `(linhas, colunas)`. `info()` mostra o tipo de cada coluna e quantos valores não nulos existem.

In [24]:
df2.info()
print('\nShape (linhas, colunas):', df2.shape)

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   cidade     4 non-null      str    
 1   populacao  4 non-null      int64  
 2   area_km2   4 non-null      float64
dtypes: float64(1), int64(1), str(1)
memory usage: 228.0 bytes

Shape (linhas, colunas): (4, 3)


### Operações na tabela inteira e gravação
Quando todas as colunas são numéricas, podemos operar na tabela inteira. Para gravar, use `to_csv` (com `index=False` para não criar uma coluna extra com o índice). O arquivo transformado vai para `outputs/`.


In [25]:
df3 = pd.read_csv('outputs/matriz_numerica.csv')
df4 = df3 * 10.3
display(df4.head(2))

df4.to_csv('outputs/matriz_transformada.csv', index=False)

,A,B,C,D
0,51.5,72.1,41.2,41.2
1,82.4,82.4,82.4,92.7


### Problemas comuns: acentuação (`encoding`) e escolha de colunas (`usecols`)

O arquivo `outputs/vendas_simuladas.csv` é gerado com codificação `cp1252` para demonstrar uma situação frequente em dados exportados de sistemas antigos. Se aparecerem símbolos estranhos ou `UnicodeDecodeError`, informe `encoding=`.

Para tabelas grandes, `usecols` carrega **somente as colunas necessárias**, por posição ou por nome.


In [26]:
try:
    pd.read_csv('outputs/vendas_simuladas.csv')           # sem informar o encoding
except UnicodeDecodeError as erro:
    print('Erro de acentuação:', str(erro)[:70], '...')

Erro de acentuação: 'utf-8' codec can't decode byte 0xed in position 2: invalid continuati ...


In [27]:
df6 = pd.read_csv('outputs/vendas_simuladas.csv', encoding='cp1252')
df6.sample(5)

,Segmento,País,Produto,Unidades,Preço,Mês,Receita
3,Pequenas empresas,França,Carretera,1456,125,9,182000
36,Pequenas empresas,França,Paseo,2048,20,12,40960
38,Pequenas empresas,Alemanha,Montana,388,35,7,13580
1,Pequenas empresas,Alemanha,Paseo,149,10,1,1490
39,Pequenas empresas,Alemanha,Montana,2777,350,12,971950


In [28]:
# Só algumas colunas, pela posição (0, 1, 2, 6)...
pd.read_csv('outputs/vendas_simuladas.csv', encoding='cp1252', usecols=[0, 1, 2, 6]).sample(5)

,Segmento,País,Produto,Receita
23,Empresas,Brasil,Carretera,68005
26,Pequenas empresas,Brasil,Paseo,150000
16,Governo,Alemanha,Carretera,58440
18,Empresas,Canadá,Montana,11380
33,Governo,Brasil,Carretera,126375


In [29]:
# ...ou pelo nome (mais seguro: não depende da ordem)
pd.read_csv('outputs/vendas_simuladas.csv', encoding='cp1252', usecols=['País', 'Receita']).sample(5)

,País,Receita
21,França,26240
31,Brasil,38660
22,França,68180
28,Alemanha,18350
19,Brasil,23125


## 7. Lendo e gravando arquivos Excel

Requer a biblioteca `openpyxl` (se necessário, instale no mesmo ambiente Python do kernel: `python -m pip install openpyxl`).

Vamos criar exemplos em `outputs/`:
- `receitas_despesas.xlsx` — tabela simples;
- `movimentacoes_financeiras.xlsx` — cabeçalho depois de linhas de título e rodapé;
- `planilha_multiplas_abas.xlsx` — arquivo com duas abas.


In [30]:
from openpyxl import Workbook

# receitas_despesas.xlsx: tabela simples
pd.DataFrame({'mes': ['jan', 'fev', 'mar', 'abr'],
              'receita': [1200, 1350, 1100, 1500],
              'despesa': [800, 900, 950, 870]}
             ).to_excel('outputs/receitas_despesas.xlsx', index=False)

# movimentacoes_financeiras.xlsx: 2 linhas de título antes do cabeçalho e 3 linhas de rodapé
wb = Workbook()
ws = wb.active
ws.append(['CONTROLE BANCÁRIO - 2026'])
ws.append([])
ws.append(['data', 'descricao', 'valor'])
for linha in [['01/10', 'Salário', 3500], ['03/10', 'Aluguel', -1200], ['07/10', 'Mercado', -450],
              ['12/10', 'Internet', -110], ['20/10', 'Freelance', 600]]:
    ws.append(linha)
ws.append([])
ws.append(['Saldo final', '', 2340])
ws.append(['Gerado automaticamente', '', ''])
wb.save('outputs/movimentacoes_financeiras.xlsx')

# planilha com duas abas
with pd.ExcelWriter('outputs/planilha_multiplas_abas.xlsx') as writer:
    pd.DataFrame({'item': ['a', 'b'], 'qtd': [1, 2]}).to_excel(writer, sheet_name='dados', index=False)
    pd.DataFrame({'banco': ['A', 'B', 'C'], 'saldo': [100, 250, 75]}).to_excel(writer, sheet_name='banco', index=False)

print('Arquivos Excel criados.')

Arquivos Excel criados.


### `read_excel`

In [31]:
df7 = pd.read_excel('outputs/receitas_despesas.xlsx')
df7.head(2)

,mes,receita,despesa
0,jan,1200,800
1,fev,1350,900


Quando a planilha tem título e rodapé, a leitura direta bagunça a tabela:

In [32]:
pd.read_excel('outputs/movimentacoes_financeiras.xlsx').head(8)

,CONTROLE BANCÁRIO - 2026,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,data,descricao,valor
2,01/10,Salário,3500
3,03/10,Aluguel,-1200
4,07/10,Mercado,-450
5,12/10,Internet,-110
6,20/10,Freelance,600
7,NaN,NaN,NaN


- `skiprows=2` → pula as 2 primeiras linhas (título e linha em branco), então o cabeçalho passa a ser a 3ª linha.
- `skipfooter=3` → ignora as 3 últimas linhas (a linha em branco e o rodapé).

In [33]:
df8 = pd.read_excel('outputs/movimentacoes_financeiras.xlsx', skiprows=2, skipfooter=3)
df8.tail(5)

,data,descricao,valor
0,01/10,Salário,3500
1,03/10,Aluguel,-1200
2,07/10,Mercado,-450
3,12/10,Internet,-110
4,20/10,Freelance,600


In [34]:
df8.to_excel('outputs/movimentacoes_tratadas.xlsx', index=False)

### Várias abas (`sheet_name`)
Por padrão o pandas lê a **primeira** aba. Use `sheet_name` para escolher uma pelo nome (ou pela posição). Com `pd.ExcelFile` você consulta quais abas existem.

In [35]:
arquivo = 'outputs/planilha_multiplas_abas.xlsx'

print('Abas do arquivo:', pd.ExcelFile(arquivo).sheet_names)

display(pd.read_excel(arquivo))                          # primeira aba
display(pd.read_excel(arquivo, sheet_name='banco'))      # aba escolhida pelo nome

Abas do arquivo: ['dados', 'banco']


,item,qtd
0,a,1
1,b,2


,banco,saldo
0,A,100
1,B,250
2,C,75


## 📌 Resumo rápido

| Tarefa | Código |
|---|---|
| Ler CSV | `pd.read_csv('arq.csv', sep=';', encoding='cp1252', usecols=[...])` |
| Ler Excel | `pd.read_excel('arq.xlsx', sheet_name='aba', skiprows=2, skipfooter=3)` |
| Gravar | `df.to_csv('arq.csv', index=False)` · `df.to_excel('arq.xlsx', index=False)` |
| Espiar | `df.head()` · `df.tail()` · `df.sample(n)` · `df.info()` · `df.shape` |
| Filtrar | `df[(df['A'] > 0) & (df['C'] > 1)]` |
| Faltantes | `df.isna().sum()` · `df.dropna()` · `df.fillna(...)` · `df.ffill()` |
| Índice | `df.set_index('col')` · `df.reset_index()` |

## ✏️ Exercícios resolvidos

Use o arquivo `outputs/vendas_simuladas.csv` (lembre do `encoding='cp1252'`):

1. Mostre quantas linhas e colunas ele tem e os tipos de cada coluna.
2. Filtre só as vendas do **Brasil** com receita acima de 20 000.
3. Descubra qual **produto** teve mais unidades vendidas (dica: `df.groupby('Produto')['Unidades'].sum()`).
4. Insira alguns `NaN` em `Unidades` (por exemplo `df.loc[[2, 5], 'Unidades'] = np.nan`) e preencha com a média da coluna.
5. Salve o resultado em um novo arquivo Excel, sem a coluna de índice.

In [36]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Produto': ['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Mouse', 'Notebook', 'Teclado', 'Monitor'],
    'País': ['Brasil', 'Brasil', 'Argentina', 'Brasil', 'Chile', 'Brasil', 'Brasil', 'Argentina'],
    'Unidades': [10, 50, 30, 12, 40, 8, 25, 15],
    'Receita': [35000, 4000, 4500, 18000, 3200, 28000, 3750, 22500],
})

In [37]:
# Exercício 1
linhas, colunas = df.shape
print(f'Linhas: {linhas}')
print(f'Colunas: {colunas}')
print(df.dtypes)

Linhas: 8
Colunas: 4
Produto       str
País          str
Unidades    int64
Receita     int64
dtype: object


In [38]:
# Exercício 2
filtro = df[(df['País'] == 'Brasil') & (df['Receita'] > 20000)]
print(filtro)

    Produto    País  Unidades  Receita
0  Notebook  Brasil        10    35000
5  Notebook  Brasil         8    28000


In [39]:
# Exercício 3
total = df.groupby('Produto')['Unidades'].sum()
print(total)
print('Mais vendido:', total.idxmax(), 'com', total.max(), 'unidades')

Produto
Monitor     27
Mouse       90
Notebook    18
Teclado     55
Name: Unidades, dtype: int64
Mais vendido: Mouse com 90 unidades


In [40]:
# Exercício 4
df.loc[[2, 5], 'Unidades'] = np.nan
print(df['Unidades'].isna().sum(), 'valores ausentes')

media = df['Unidades'].mean()
df['Unidades'] = df['Unidades'].fillna(media)
print(df)

2 valores ausentes
    Produto       País   Unidades  Receita
0  Notebook     Brasil  10.000000    35000
1     Mouse     Brasil  50.000000     4000
2   Teclado  Argentina  25.333333     4500
3   Monitor     Brasil  12.000000    18000
4     Mouse      Chile  40.000000     3200
5  Notebook     Brasil  25.333333    28000
6   Teclado     Brasil  25.000000     3750
7   Monitor  Argentina  15.000000    22500


In [41]:
# Exercício 5
from pathlib import Path

# Define a pasta de saída
pasta_outputs = Path("outputs")

# Cria a pasta caso não exista
pasta_outputs.mkdir(exist_ok=True)

# Salva o DataFrame em Excel
df.to_excel(pasta_outputs / "resultado.xlsx", index=False)

print("Arquivo salvo em:", (pasta_outputs / "resultado.xlsx").resolve())

Arquivo salvo em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/01_Programacao_Python/outputs/resultado.xlsx


## Aprofundamento 1 — Contratos e validação de dados
Antes de analisar, confirme colunas obrigatórias, tipos e restrições essenciais. Isso evita resultados aparentemente plausíveis baseados em dados incorretos.

In [42]:
import pandas as pd
def validar_vendas(df: pd.DataFrame) -> pd.DataFrame:
    obrigatorias = {"produto", "quantidade", "preco"}
    ausentes = obrigatorias - set(df.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes: {sorted(ausentes)}")
    if df[["quantidade", "preco"]].isna().any().any():
        raise ValueError("Dados numéricos incompletos")
    if (df[["quantidade", "preco"]] < 0).any().any():
        raise ValueError("Quantidades e preços devem ser não negativos")
    return df.copy()

vendas_exemplo = pd.DataFrame({"produto": ["A", "B"], "quantidade": [2, 3], "preco": [10., 7.]})
assert validar_vendas(vendas_exemplo).shape == (2, 3)
print("Contrato de dados: OK")

Contrato de dados: OK


## Aprofundamento 2 — `merge(validate=...)` e cardinalidade
Duplicações de chaves podem multiplicar linhas inesperadamente. O parâmetro `validate` torna a hipótese de cardinalidade explícita.

In [43]:
pedidos = pd.DataFrame({"pedido": [1, 2, 3], "produto_id": [10, 20, 10]})
catalogo = pd.DataFrame({"produto_id": [10, 20], "categoria": ["A", "B"]})
resultado = pedidos.merge(catalogo, on="produto_id", how="left", validate="many_to_one", indicator=True)
assert len(resultado) == len(pedidos)
assert resultado["_merge"].eq("both").all()
print(resultado)

   pedido  produto_id categoria _merge
0       1          10         A   both
1       2          20         B   both
2       3          10         A   both


## Aprofundamento 3 — Evitando cópias ambíguas e vazamento temporal
Para atualizar subconjuntos, use `.loc` ou `.assign()`. Em séries temporais, não use registros posteriores ao momento previsto para calcular atributos de treino.

In [44]:
tabela = pd.DataFrame({"produto": ["A", "B", "C"], "preco": [10., 20., 30.]})
limpo = tabela.assign(preco_com_imposto=lambda d: d["preco"] * 1.10)
assert "preco_com_imposto" not in tabela.columns
assert limpo.loc[0, "preco_com_imposto"] == 11.0
print(limpo)

  produto  preco  preco_com_imposto
0       A   10.0               11.0
1       B   20.0               22.0
2       C   30.0               33.0


### Exercício com solução
Agregue o faturamento por produto, garantindo que a quantidade de linhas do catálogo original não seja alterada por uma operação de junção.

In [45]:
transacoes = pd.DataFrame({"produto": ["A", "B", "A"], "quantidade": [1, 2, 3], "preco": [5., 7., 5.]})
faturamento = (transacoes.assign(valor=lambda d: d.quantidade * d.preco)
                .groupby("produto", as_index=False)["valor"].sum())
assert faturamento.set_index("produto").loc["A", "valor"] == 20
print(faturamento)

  produto  valor
0       A   20.0
1       B   14.0
